# Benchmarking Mechanistic Interpretability Methods Across Open-Source Models

This notebook is a **comparison harness**, not a tour. It runs a fixed set of interpretability methods
over a fixed set of tasks and models, scores them on common metrics, and produces tables and plots that
answer: *which method should I use, on which model, and what does it cost me?*

## The central problem: "compared on what?"

Interpretability methods are not interchangeable, and comparing them naively is a category error. Asking
whether the logit lens beats activation patching is like asking whether a thermometer beats a scale. So this
notebook splits methods into **families**, and only compares *within* a family:

**Family A — component attribution.** Every method here answers the same question: *how important is
attention head (L, H) for this behaviour?* and returns a score per head. These are directly comparable.

| Method | Type | Cost (forward passes) |
|---|---|---|
| Activation patching | causal, exact | `n_layers × n_heads` |
| Zero ablation | causal, off-distribution | `n_layers × n_heads` |
| Mean ablation | causal, on-distribution | `n_layers × n_heads` |
| Attribution patching | causal, linear approximation | 2 fwd + 2 bwd |
| Direct logit attribution | linear decomposition, direct effect only | 1 |
| Attention-to-answer | correlational baseline | 1 |
| Random | control | 0 |

**Family B — representation decoding.** *Where and when does the model encode the answer?* Compared across
models by depth-of-emergence, not against each other for accuracy.

We evaluate Family A two ways, because each is flawed alone:

1. **Agreement with exhaustive activation patching** (Spearman ρ, top-k overlap, sign agreement). Patching is
   the reference — but it is a *reference, not ground truth*. Single-head patching systematically
   underrates redundant components, so any method that agrees with it inherits that blind spot.
2. **Faithfulness curves**, which privilege no method: take each method's top-k heads, mean-ablate
   everything else, and measure how much task performance survives. This is the metric that actually
   matters if you're using a method to *find a circuit*, and it needs no reference ranking at all.

Plus **cost**, measured in forward/backward passes and wall-clock, giving a **fidelity–cost Pareto plot**.
The interesting result is usually not which method is best, but how much fidelity you give up for a 100×
speedup — and whether that trade-off changes with model architecture.

> **Runtime:** the full run below took roughly 15 minutes on an A100. Five of six models completed; only
> SmolLM3-3B failed (see §5 for why, and why it is not the memory leak that broke earlier runs).

## Headline result

Across **five architectures** and **two tasks**, agreement with exhaustive activation patching, restricted
to the 32 heads the reference calls important:

| method | passes | GPT-2 | Pythia-410M | Qwen3-0.6B | OLMo-2-1B | Gemma-3-1B |
|---|---|---|---|---|---|---|
| **attribution patching** | **4** | **0.990** | **0.945** | **0.876** | **0.923** | **0.908** |
| direct logit attribution | 1 | 0.760 | 0.686 | 0.732 | 0.643 | 0.688 |
| mean ablation | n·h+1 | 0.538 | 0.541 | 0.587 | 0.466 | 0.363 |
| zero ablation | n·h+1 | 0.283 | 0.377 | 0.391 | 0.499 | 0.365 |
| attention to answer | 1 | 0.310 | 0.163 | 0.040 | 0.191 | 0.179 |
| random | 0 | −0.029 | −0.094 | 0.044 | 0.001 | 0.052 |

**Attribution patching reproduces exhaustive patching at ρ = 0.88–0.99 for four model passes instead of
105–449** — a 26× to 112× saving depending on model shape, and it holds across MHA, GQA, RMSNorm, post-norm
and sliding-window architectures. If you take one thing from this notebook: screen with attribution
patching, then verify the contested handful exactly.

But the two evaluation axes **disagree about which method is best**, and that turns out to be the more
interesting finding — see §6.

## 0. Setup

TransformerLens 3 requires `transformers >= 5.4`; Colab will probably ask you to restart after installing.
Restart, then resume at the next cell.

In [ ]:
%pip install -q "transformer_lens>=3.0" "sae_lens>=6.0" plotly einops scikit-learn scipy pandas huggingface_hub
print("If Colab shows RESTART SESSION, click it, then continue from the next cell.")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.7/334.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.7/241.7 kB 26.0 MB/s eta 0:00:00
If Colab shows RESTART SESSION, click it, then continue from the next cell.


In [ ]:
import os
# reduce fragmentation; must be set before torch initialises CUDA
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, time, math, random, itertools, warnings
from functools import partial
from dataclasses import dataclass, field
from typing import Callable, Optional

import torch, torch.nn as nn
import numpy as np
import pandas as pd
import einops
from scipy.stats import spearmanr
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.auto import tqdm

import transformer_lens
from transformer_lens.model_bridge import TransformerBridge
from transformer_lens import utils

try:
    import google.colab  # noqa
    pio.renderers.default = "colab"
except ImportError:
    pass

torch.set_grad_enabled(False)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
#print("transformer_lens", transformer_lens.__version__, "| device", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token:", "present (gated models available)" if HF_TOKEN else "absent (ungated models only)")

def purge():
    # NOTE: a `free(model)` helper CANNOT work - `del` inside a function only drops that
    # function's local reference. Memory is released by letting scopes die (see run_one_model).
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def gpu_mem():
    if not torch.cuda.is_available():
        return "cpu"
    g = 2 ** 30
    return (f"{torch.cuda.memory_allocated()/g:.2f} GiB alloc / "
            f"{torch.cuda.memory_reserved()/g:.2f} reserved / "
            f"{torch.cuda.get_device_properties(0).total_memory/g:.1f} total")

/tmp/ipykernel_4896/1369129068.py:22: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import utils


NVIDIA A100-SXM4-40GB
HF token: present (gated models available)


## 1. Benchmark configuration

Models are chosen to vary the things that plausibly affect method behaviour: **depth, width, attention
scheme (MHA vs GQA), normalisation, and training corpus**. GPT-2 small is included deliberately as a
control — it is the model every one of these methods was developed and validated on, so if the harness
disagrees with published IOI results there, the harness is wrong, not the model.

**What loaded in the final run.** All five completed and all five performed both tasks:

| model | layers × heads | kv heads | patching passes/task | note |
|---|---|---|---|---|
| GPT-2 small | 12 × 12 | 12 | 145 | control; MHA, LayerNorm |
| Pythia-410M | 24 × 16 | 16 | 385 | needed the v4 attribute alias to load at all |
| Qwen3-0.6B | 28 × 16 | 8 | 449 | GQA 2:1, RMSNorm, no BOS |
| OLMo-2-1B | 16 × 16 | 16 | 257 | post-norm; **`fold_ln` unsupported** |
| Gemma-3-1B | 26 × 4 | 1 | 105 | GQA 4:1, sliding-window attention |

The `passes/task` column is worth pausing on: it varies 4× across models of similar size, because
exhaustive patching costs `n_layers × n_heads` and Gemma-3 has only 4 heads per layer while Qwen3 has 16.
Cost comparisons between methods are therefore model-shape-dependent, not a single number.

In [ ]:
QUICK = False         # True -> 4 prompts/task instead of 8 (never subsamples layers)

@dataclass
class ModelSpec:
    hf_name: str
    label: str
    params_b: float = 0.5          # rough parameter count, used to pick a dtype
    dtype: Optional[torch.dtype] = None   # None -> chosen automatically for this GPU
    gated: bool = False
    note: str = ""

def pick_dtype(params_b):
    # T4 (16 GB, Turing) has NO native bf16. A100/L4/H100 do.
    if not torch.cuda.is_available():
        return torch.float32
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    bf16 = torch.cuda.is_bf16_supported()
    # fp32 needs ~4 GB/B params for weights, plus activations and (for attribution
    # patching) gradients. Budget ~3x weights.
    if params_b * 4 * 3 < total * 0.6:
        return torch.float32
    return torch.bfloat16 if bf16 else torch.float16

MODEL_SPECS = [
    ModelSpec("openai-community/gpt2",    "GPT-2 small", 0.12, note="control: MHA, LayerNorm, learned pos"),
    ModelSpec("EleutherAI/pythia-410m",   "Pythia-410M", 0.41, note="MHA, parallel attn+MLP"),
    ModelSpec("Qwen/Qwen3-0.6B",          "Qwen3-0.6B",  0.60, note="GQA, RMSNorm, RoPE, no BOS"),
    ModelSpec("allenai/OLMo-2-0425-1B",   "OLMo-2-1B",   1.48, note="post-norm; fold_ln unsupported"),
    ModelSpec("google/gemma-3-1b-pt",     "Gemma-3-1B",  1.00, gated=True, note="sliding-window attention"),
    ModelSpec("HuggingFaceTB/SmolLM3-3B", "SmolLM3-3B",  3.08, note="3B, GQA"),
]
MODEL_SPECS = [m for m in MODEL_SPECS if (HF_TOKEN is not None or not m.gated)]
for m in MODEL_SPECS:
    if m.dtype is None:
        m.dtype = pick_dtype(m.params_b)

N_PROMPTS = 4 if QUICK else 8
print(f"GPU: {gpu_mem()}")
print(f"{len(MODEL_SPECS)} models, {N_PROMPTS} prompts/task, QUICK={QUICK}\n")
for m in MODEL_SPECS:
    print(f"  {m.label:<14}{str(m.dtype).replace('torch.',''):<10}{m.hf_name:<32}{m.note}")

GPU: 0.00 GiB alloc / 0.00 reserved / 39.5 total
6 models, 8 prompts/task, QUICK=False

  GPT-2 small   float32   openai-community/gpt2           control: MHA, LayerNorm, learned pos
  Pythia-410M   float32   EleutherAI/pythia-410m          MHA, parallel attn+MLP
  Qwen3-0.6B    float32   Qwen/Qwen3-0.6B                 GQA, RMSNorm, RoPE, no BOS
  OLMo-2-1B     float32   allenai/OLMo-2-0425-1B          post-norm; fold_ln unsupported
  Gemma-3-1B    float32   google/gemma-3-1b-pt            sliding-window attention
  SmolLM3-3B    bfloat16  HuggingFaceTB/SmolLM3-3B        3B, GQA


In [ ]:
LOAD_NOTES, FOLD_LN_OK = {}, {}

def _hf_model_with_aliases(spec):
    # transformers v5 renamed several LM heads (GPTNeoX: embed_out -> lm_head), but some
    # TransformerLens adapters still look for the old name. Re-expose it on the instance;
    # assigning a Module under a second attribute shares it, it does not copy weights.
    from transformers import AutoModelForCausalLM
    hf = AutoModelForCausalLM.from_pretrained(
        spec.hf_name, dtype=spec.dtype, attn_implementation="eager").to(DEVICE)   # <- must match
    # (without .to(DEVICE) the bridge runs on cuda while these weights sit on cpu, and the
    #  forward fails with "Expected all tensors to be on the same device")
    for old, new in [("embed_out", "lm_head"), ("gpt_neox", "model")]:
        if not hasattr(hf, old) and hasattr(hf, new):
            setattr(hf, old, getattr(hf, new))
    return hf

def load_model(spec, compat=True):
    notes = []
    attempts = [
        ("eager attention", lambda: TransformerBridge.boot_transformers(
            spec.hf_name, device=DEVICE, dtype=spec.dtype,
            hf_config_overrides={"attn_implementation": "eager"})),
        ("default", lambda: TransformerBridge.boot_transformers(
            spec.hf_name, device=DEVICE, dtype=spec.dtype)),
        ("pre-loaded HF model with v4 attribute aliases", lambda: TransformerBridge.boot_transformers(
            spec.hf_name, hf_model=_hf_model_with_aliases(spec), device=DEVICE, dtype=spec.dtype)),
    ]
    m, last_msg = None, None
    for label, fn in attempts:
        try:
            m = fn(); notes.append(f"loaded via: {label}"); break
        except Exception as e:
            # CRITICAL: keeping the exception object alive keeps its traceback alive, which keeps
            # the frames of the failed attempt alive, which keeps the PARTIALLY LOADED MODEL alive.
            # purge() then frees nothing and each failed attempt leaks a whole model.
            last_msg = f"{type(e).__name__}: {str(e)[:200]}"
            notes.append(f"  {label} failed -> {last_msg[:130]}")
            e.__traceback__ = None
            del e
            purge()
    if m is None:
        LOAD_NOTES[spec.label] = notes
        for n in notes: print("   ", n)
        raise RuntimeError(f"all load strategies failed; last error: {last_msg}")

    fold_ln_ok = True
    if compat:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            m.enable_compatibility_mode()   # fold_ln + centre weights
        for w in caught:
            if "fold_ln" in str(w.message):
                fold_ln_ok = False
                notes.append("fold_ln NOT supported by this adapter -> "
                             "direct_logit_attr and the logit lens are unreliable for this model")
    m.eval()
    LOAD_NOTES[spec.label] = notes
    FOLD_LN_OK[spec.label] = fold_ln_ok
    for n in notes:
        print("   ", n)
    return m

### 1.1 Preflight

A model only enters the benchmark if it passes. This matters more here than in a tutorial: a silently
missing `hook_pattern`, or a residual decomposition that doesn't hold because compatibility mode failed,
would produce plausible-looking numbers for every method and invalidate the whole comparison.

In [ ]:
def preflight(model, label):
    issues = []
    cfg = model.cfg
    try:
        toks = model.to_tokens("The Eiffel Tower is in the city of")
        logits, cache = model.run_with_cache(toks)
    except Exception as e:
        return [f"forward pass failed: {e}"], None

    L = min(1, cfg.n_layers - 1)
    for key, desc in [("resid_pre", "residual stream"), ("z", "per-head output"),
                      ("pattern", "attention pattern"), ("attn_out", "attn block out"),
                      ("mlp_out", "mlp block out")]:
        try:
            _ = cache[key, L]
        except Exception:
            issues.append(f"missing hook: {key}")

    try:
        err = (cache["resid_post", L] - (cache["resid_pre", L] + cache["attn_out", L]
               + cache["mlp_out", L])).abs().max().item()
        if err > 1e-2:
            issues.append(f"residual decomposition violated (max err {err:.3f})")
    except Exception as e:
        issues.append(f"could not check residual decomposition: {e}")

    try:
        z = cache["z", L]
        if z.shape[-2] != cfg.n_heads:
            issues.append(f"z has {z.shape[-2]} heads, cfg says {cfg.n_heads}")
    except Exception:
        pass

    info = dict(n_layers=cfg.n_layers, n_heads=cfg.n_heads, d_model=cfg.d_model,
                n_kv_heads=getattr(cfg, "n_key_value_heads", None) or cfg.n_heads,
                top_pred=model.to_string(logits[0, -1].argmax()))
    return issues, info

In [ ]:
def get_W_U(model):
    v = model.cfg.d_vocab
    try:
        W = model.W_U
        if W.shape[-1] == v:
            return W.detach()
    except Exception:
        pass
    for mod in model.modules():
        if isinstance(mod, nn.Linear) and mod.out_features == v:
            return mod.weight.detach().T
    for mod in model.modules():
        if isinstance(mod, nn.Embedding) and mod.num_embeddings == v:
            return mod.weight.detach().T          # tied embeddings
    raise RuntimeError("cannot locate unembedding matrix")

def single_token_names(model):
    cands = [" John", " Mary", " Tom", " James", " Dan", " Sid", " Martin", " Amy", " Alice", " Bob",
             " Sarah", " David", " Emma", " Peter", " Anna", " Paul", " Lucy", " Mark", " Kate", " Sam",
             " Michael", " Laura", " Chris", " Julia", " Simon", " Rachel", " Kevin", " Nina"]
    out = []
    for n in cands:
        try:
            if model.to_tokens(n, prepend_bos=False).shape[1] == 1 and n not in out:
                out.append(n)
        except Exception:
            pass
    return out

## 2. Task battery

Each task supplies **clean tokens, corrupted tokens, and a scalar metric** with the property that
clean ≫ corrupted. Methods are then compared on how well they localise the components responsible for
that gap. Two tasks, chosen because they stress different circuitry:

* **IOI** — *"When John and Mary went to the store, Mary gave a drink to"* → `" John"`. A multi-component
  circuit (duplicate-token → S-inhibition → name-mover). Corruption swaps the subject, flipping the answer.
* **Induction** — a repeated random token sequence. The model must copy what followed the earlier
  occurrence. Corruption replaces the first copy with different random tokens, destroying the pattern while
  keeping the second half (and therefore all positions and the final query) identical.

Two tasks matter because a method can look excellent on one circuit and poor on another; a single-task
benchmark would be a claim about that circuit, not about the method.

**Measured behavioural gaps** (clean − corrupt logit difference; the gate requires > 0.5):

| model | IOI | Induction |
|---|---|---|
| GPT-2 small | 6.50 | 17.08 |
| Pythia-410M | 6.74 | 16.37 |
| Qwen3-0.6B | 8.54 | 23.50 |
| OLMo-2-1B | 11.22 | 19.08 |
| Gemma-3-1B | 11.38 | 19.51 |

Every model cleared the gate on both tasks, so all ten (model, task) pairs entered the benchmark. Induction
gaps are 2–3× larger than IOI gaps throughout — induction is a much easier, more sharply localised
behaviour, which is worth remembering when method scores are lower on it (see §6.1's per-task tables).

In [ ]:
@dataclass
class Task:
    name: str
    clean_tokens: torch.Tensor
    corrupt_tokens: torch.Tensor
    str_tokens: list
    answer_ids: torch.Tensor     # correct next token, per prompt
    control_ids: torch.Tensor    # the wrong alternative the metric contrasts against
    answer_pos: torch.Tensor     # key position a head should attend to (for the correlational baseline)
    clean_score: float = 0.0
    corrupt_score: float = 0.0

    def metric(self, logits):
        f = logits[:, -1]
        return (f.gather(1, self.answer_ids[:, None]) - f.gather(1, self.control_ids[:, None])).mean()

    def normalise(self, v):
        d = self.clean_score - self.corrupt_score
        return (v - self.corrupt_score) / d if abs(d) > 1e-6 else float("nan")

    def answer_dirs(self, W_U):
        # residual-space direction that raises the metric: W_U[:,answer] - W_U[:,control]
        return (W_U[:, self.answer_ids] - W_U[:, self.control_ids]).T.float()


def build_ioi_task(model, n=N_PROMPTS, seed=0):
    names = single_token_names(model)
    if len(names) < 4:
        raise RuntimeError(f"only {len(names)} single-token names for this tokenizer")
    TPL = "When{A} and{B} went to the store,{S} gave a drink to"
    rng = random.Random(seed)
    clean, corrupt, io, s_ = [], [], [], []
    for _ in range(n):
        a, b = rng.sample(names, 2)
        clean.append(TPL.format(A=a, B=b, S=b)); io.append(a); s_.append(b)
        corrupt.append(TPL.format(A=a, B=b, S=a))      # subject swaps -> answer flips
    ct, pt = model.to_tokens(clean), model.to_tokens(corrupt)
    if ct.shape != pt.shape:
        raise RuntimeError("clean/corrupt token lengths differ - name tokenisation issue")
    tok = lambda x: model.to_tokens(x, prepend_bos=False)[0, 0].item()
    io_ids = torch.tensor([tok(x) for x in io], device=DEVICE)
    s_ids = torch.tensor([tok(x) for x in s_], device=DEVICE)
    # position of the IO name's first mention: where a name-mover head should look
    pos = []
    for i in range(ct.shape[0]):
        hit = (ct[i] == io_ids[i]).nonzero()
        pos.append(int(hit[0]) if len(hit) else 0)
    return Task("IOI", ct, pt, model.to_str_tokens(clean[0]),
                io_ids, s_ids, torch.tensor(pos, device=DEVICE))


def build_induction_task(model, n=N_PROMPTS, seq_len=None, seed=0):
    seq_len = seq_len or (16 if QUICK else 24)
    g = torch.Generator().manual_seed(seed)
    lo, hi = 1000, min(model.cfg.d_vocab, 40000) - 1000
    r1 = torch.randint(lo, hi, (n, seq_len), generator=g)
    r2 = torch.randint(lo, hi, (n, seq_len), generator=g)      # unrelated first half
    bos_id = getattr(model.tokenizer, "bos_token_id", None)
    b = 1 if bos_id is not None else 0
    def assemble(first, second):
        parts = ([torch.full((n, 1), bos_id)] if b else []) + [first, second]
        return torch.cat(parts, 1).to(DEVICE)

    # second copy truncated by one, so the final position's correct continuation is r1[:, -1]
    clean = assemble(r1, r1[:, :-1])
    corrupt = assemble(r2, r1[:, :-1])          # same second half, same positions, no valid pattern
    answer = r1[:, -1].to(DEVICE)
    control = r2[:, -1].to(DEVICE)
    # an induction head at the final query should attend to the token AFTER the earlier occurrence
    pos = torch.full((n,), b + seq_len - 1, device=DEVICE)
    return Task("Induction", clean, corrupt, [str(t) for t in clean[0].tolist()],
                answer, control, pos)


TASK_BUILDERS = {"IOI": build_ioi_task, "Induction": build_induction_task}

In [ ]:
def score_task(model, task):
    task.clean_score = task.metric(model(task.clean_tokens)).item()
    task.corrupt_score = task.metric(model(task.corrupt_tokens)).item()
    return task

def task_is_usable(task, min_gap=0.5):
    # Methods can only be compared where there is a real behavioural gap to explain.
    return (task.clean_score - task.corrupt_score) > min_gap and task.clean_score > 0

## 3. The method zoo

Each method is a function `(model, task) -> [n_layers, n_heads]` of importance scores, wrapped so that
**cost is measured, not assumed**. Sign convention throughout: **higher = more important for producing the
correct answer**, so ablation scores are negated (removing a useful head lowers the metric).

All methods operate on `z`, the per-head output before `W_O`. This is the right granularity for grouped-query
attention: GQA shares keys and values across heads but each query head still has its own `z`.

In [ ]:
class Counter:
    def __init__(self): self.fwd = 0; self.bwd = 0; self.t0 = time.time()
    def done(self): return dict(fwd=self.fwd, bwd=self.bwd, seconds=time.time() - self.t0)

def head_grid(model): return torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

def layers_of(model):
    # Every method must cover the SAME components, or the comparison is meaningless.
    # QUICK therefore shrinks prompts/sequences, never the layer set.
    return list(range(model.cfg.n_layers))

In [ ]:
# ---------- 1. Activation patching (exact, the reference) ----------
def m_activation_patching(model, task, c: Counter):
    src = model.run_with_cache(task.clean_tokens)[1]
    c.fwd += 1
    out = head_grid(model)
    def hook(act, hook, head):
        act[:, :, head] = src[hook.name][:, :, head]
        return act
    for L in tqdm(layers_of(model), desc="  activation patching", leave=False):
        for H in range(model.cfg.n_heads):
            lg = model.run_with_hooks(task.corrupt_tokens,
                    fwd_hooks=[(utils.get_act_name("z", L), partial(hook, head=H))])
            c.fwd += 1
            out[L, H] = task.normalise(task.metric(lg).item())
    return out

# ---------- 2/3. Ablation ----------
def _ablation(model, task, c, mode):
    ref = model.run_with_cache(task.clean_tokens)[1]
    c.fwd += 1
    out = head_grid(model)
    def hook(act, hook, head):
        if mode == "zero":
            act[:, :, head] = 0.0
        else:
            act[:, :, head] = ref[hook.name][:, :, head].mean(0, keepdim=True)
        return act
    for L in tqdm(layers_of(model), desc=f"  {mode} ablation", leave=False):
        for H in range(model.cfg.n_heads):
            lg = model.run_with_hooks(task.clean_tokens,
                    fwd_hooks=[(utils.get_act_name("z", L), partial(hook, head=H))])
            c.fwd += 1
            # importance = how much performance DROPS when removed
            out[L, H] = (task.clean_score - task.metric(lg).item()) / (task.clean_score - task.corrupt_score)
    return out

def m_zero_ablation(model, task, c): return _ablation(model, task, c, "zero")
def m_mean_ablation(model, task, c): return _ablation(model, task, c, "mean")

# ---------- 4. Attribution patching (gradient approximation) ----------
def m_attribution_patching(model, task, c: Counter):
    is_z = lambda n: n.endswith("hook_z")
    def fwd_bwd(tokens):
        # TransformerLens calls hooks as fn(tensor, hook=<HookPoint>) - `hook` is a KEYWORD
        # argument, so the second parameter must literally be named `hook`.
        acts, grads = {}, {}
        model.reset_hooks()
        try:
            model.add_hook(is_z, lambda a, hook: acts.__setitem__(hook.name, a.detach()), "fwd")
            model.add_hook(is_z, lambda g, hook: grads.__setitem__(hook.name, g.detach()), "bwd")
            with torch.set_grad_enabled(True):
                task.metric(model(tokens)).backward()
        finally:
            model.reset_hooks()      # must run even on failure, else bad hooks poison later methods
            model.zero_grad(set_to_none=True)   # backward() allocates .grad on EVERY parameter,
                                                # which doubles resident memory if left in place
        c.fwd += 1; c.bwd += 1
        return acts, grads

    corrupt_acts, corrupt_grads = fwd_bwd(task.corrupt_tokens)
    clean_acts, _ = fwd_bwd(task.clean_tokens)
    out = head_grid(model)
    for L in range(model.cfg.n_layers):
        n = utils.get_act_name("z", L)
        if n not in corrupt_grads:
            continue
        delta = (clean_acts[n] - corrupt_acts[n]).float()
        out[L] = (corrupt_grads[n].float() * delta).sum(dim=(0, 1, 3)).cpu()
    return out / (task.clean_score - task.corrupt_score)

# ---------- 5. Direct logit attribution ----------
def m_direct_logit_attribution(model, task, c: Counter, W_U=None):
    _, cache = model.run_with_cache(task.clean_tokens)
    c.fwd += 1
    answer_dirs = task.answer_dirs(W_U)
    final = cache[f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"][:, -1].float()
    try:
        scale = cache["ln_final.hook_scale"][:, -1].float()
        if scale.ndim == 1: scale = scale[:, None]
    except Exception:
        scale = final.norm(dim=-1, keepdim=True) / math.sqrt(model.cfg.d_model)
    W_O = model.W_O.float()
    out = head_grid(model)
    for L in range(model.cfg.n_layers):
        z = cache["z", L][:, -1].float()
        contrib = einops.einsum(z, W_O[L], "b h d, h d m -> b h m") / scale[:, None, :]
        out[L] = einops.einsum(contrib, answer_dirs, "b h m, b m -> b h").mean(0).cpu()
    return out / (task.clean_score - task.corrupt_score)

# ---------- 6. Correlational baseline: attention to the answer token ----------
def m_attention_to_answer(model, task, c: Counter, W_U=None):
    _, cache = model.run_with_cache(task.clean_tokens)
    c.fwd += 1
    out = head_grid(model)
    answer_pos = task.answer_pos
    for L in range(model.cfg.n_layers):
        pat = cache["pattern", L].float()                 # [b, head, q, k]
        final_q = pat[:, :, -1, :]                        # attention FROM the last token
        out[L] = final_q.gather(2, answer_pos.view(-1, 1, 1).expand(-1, model.cfg.n_heads, 1)
                                ).squeeze(-1).mean(0).cpu()
    return out

# ---------- 7. Random control ----------
def m_random(model, task, c: Counter):
    g = torch.Generator().manual_seed(0)
    return torch.randn(model.cfg.n_layers, model.cfg.n_heads, generator=g)

### Why include a random control and a correlational baseline?

Because without them, "method X gets ρ = 0.4 against patching" is uninterpretable. The random control fixes
the floor. The attention-to-answer baseline is more pointed: it is what you get from *looking at attention
patterns*, the most common informal interpretability move. If a causal method doesn't clearly beat it, the
extra compute bought nothing — and if it does, that's a concrete argument for causal methods over
attention-staring.

**The answer, from the run.** The random control lands at −0.09 to +0.05 rank agreement and 0.00 normalised
faithfulness by construction, so the floor is where it should be. The attention baseline scores 0.04–0.31
on rank agreement and −0.06 to +0.32 on normalised faithfulness — better than random on some models,
*worse* on two. So reading attention patterns is not a reliable substitute for a causal method, and having
both baselines is what turns "attribution patching scores 0.9" from a number into a claim.

In [ ]:
METHODS = {
    "activation_patching":  dict(fn=m_activation_patching,     exact=True,  kind="causal"),
    "mean_ablation":        dict(fn=m_mean_ablation,           exact=False, kind="causal"),
    "zero_ablation":        dict(fn=m_zero_ablation,           exact=False, kind="causal"),
    "attribution_patching": dict(fn=m_attribution_patching,    exact=False, kind="gradient"),
    "direct_logit_attr":    dict(fn=m_direct_logit_attribution, exact=False, kind="linear", needs_W_U=True),
    "attention_to_answer":  dict(fn=m_attention_to_answer,     exact=False, kind="correlational", needs_W_U=True),
    "random":               dict(fn=m_random,                  exact=False, kind="control"),
}
REFERENCE = "activation_patching"
print("\n".join(f"{k:<22} {v['kind']}" for k, v in METHODS.items()))

activation_patching    causal
mean_ablation          causal
zero_ablation          causal
attribution_patching   gradient
direct_logit_attr      linear
attention_to_answer    correlational
random                 control


## 4. Evaluation metrics

### 4.1 Agreement with the reference
`spearman` (rank correlation over all heads), `top_k_overlap` (Jaccard of the top-k head sets by |score|),
and `sign_agreement` on the reference's important heads. Rank metrics are the right choice: methods return
scores on incompatible scales, and what you actually use a method for is *ranking* components.

### 4.2 Faithfulness — the reference-free metric
Take a method's top-k heads, **mean-ablate every other head in the model**, and measure how much of the
clean–corrupt gap survives. A method whose top-k really is the circuit will recover most of the performance
at small k. This makes no assumption about which method is correct, which is why it carries more weight
here than agreement with patching.

We summarise each curve by its normalised area (AUC), so each (model, task, method) gets one number.

Note the scope: only **attention heads** are ablated, MLPs are left intact. So "recovery" here means
"recovery given the MLPs", and a method cannot be penalised for MLP-mediated effects it was never asked
about. Extending the same harness to neurons is the natural next step (§9) and is where the cheap methods
should pull decisively ahead.

**Two refinements added after the first full run**, both because the raw metrics proved misleading:

* `spearman_topk` restricts rank agreement to the reference's 32 most important heads. All-head Spearman is
  dominated by the ~90% of heads with no effect, whose ordering is arbitrary in every method.
* `auc_norm` normalises faithfulness against the random control per (model, task). Raw AUC has a floor of
  0.49–0.85 because ablating attention heads leaves every MLP intact, which made a random ranking look
  respectable and a mediocre method look excellent.

In [ ]:
def agreement_metrics(scores, reference, k=10, k_rank=32):
    a = scores.flatten().numpy(); b = reference.flatten().numpy()
    if np.isnan(a).any() or np.isnan(b).any():
        return dict(spearman=np.nan, spearman_topk=np.nan, top_k_overlap=np.nan, sign_agreement=np.nan)
    rho = spearmanr(a, b).correlation
    # Spearman over ALL heads is dominated by the ~90% of heads with no effect, whose relative
    # order is pure noise in every method. Restricting to the reference's most important heads
    # asks the question we actually care about: do the methods agree about what matters?
    imp = np.argsort(-np.abs(b))[:k_rank]
    rho_top = spearmanr(a[imp], b[imp]).correlation
    ka = set(np.argsort(-np.abs(a))[:k].tolist())
    kb = set(np.argsort(-np.abs(b))[:k].tolist())
    idx = list(kb)
    return dict(spearman=rho, spearman_topk=rho_top,
                top_k_overlap=len(ka & kb) / len(ka | kb),
                sign_agreement=float(np.mean(np.sign(a[idx]) == np.sign(b[idx]))))

In [ ]:
def faithfulness_curve(model, task, scores, ks=None):
    # Keep the top-k heads by |score|; mean-ablate all others. Returns (ks, normalised recovery).
    if torch.isnan(scores).any():
        return [], []
    n_heads_total = model.cfg.n_layers * model.cfg.n_heads
    ks = ks or [k for k in [1, 2, 4, 8, 16, 32, 64, 128] if k < n_heads_total] + [n_heads_total]
    ref_cache = model.run_with_cache(task.clean_tokens)[1]
    order = torch.argsort(scores.flatten().abs(), descending=True)

    recoveries = []
    for k in ks:
        keep = set(order[:k].tolist())
        def hook(act, hook, layer):          # 2nd arg MUST be named `hook` (passed as a kwarg)
            for H in range(model.cfg.n_heads):
                if layer * model.cfg.n_heads + H not in keep:
                    act[:, :, H] = ref_cache[hook.name][:, :, H].mean(0, keepdim=True)
            return act
        hooks = [(utils.get_act_name("z", L), partial(hook, layer=L)) for L in range(model.cfg.n_layers)]
        lg = model.run_with_hooks(task.clean_tokens, fwd_hooks=hooks)
        recoveries.append(task.normalise(task.metric(lg).item()))
    return ks, recoveries

def curve_auc(ks, recoveries, n_total):
    # normalised area under recovery-vs-log(k); 1.0 = perfect recovery at k=1
    if not ks: return np.nan
    x = np.log2(np.array(ks)); y = np.clip(np.array(recoveries), 0, 1.2)
    trapz = getattr(np, "trapezoid", np.trapz)   # np.trapz deprecated in numpy 2
    return float(trapz(y, x) / (x[-1] - x[0]))

### Hardware and the memory question, resolved

Earlier runs of this notebook failed with OOM on three models. That was **a bug here, not a capacity
limit**: `load_model` stored each failed attempt's exception, and holding an exception holds its
`__traceback__`, which holds the frames of the failed attempt, which holds the **partially loaded model**.
`purge()` then freed nothing and every fallback attempt leaked a whole model.

**The fix works, and the run log proves it.** Memory reads `0.02 GiB` before each of the five models and
`0.02 GiB` again after SmolLM3-3B's failure — the failed load released cleanly instead of leaving 13.85 GiB
stranded as it did before.

**SmolLM3-3B's remaining OOM is genuine, and an A100 did not solve it.** It exhausted all 39.5 GiB during
loading. A 3.08B model in bf16 is about 6 GiB of weights, so consuming 39 GiB points at the load path
materialising several full-precision copies — most likely `enable_compatibility_mode()`, which folds norms
and re-centres the writing weights and unembedding, and has to build new tensors to do it. Three things to
try, cheapest first: force `dtype=torch.bfloat16` on the spec rather than relying on `pick_dtype`; load with
`compat=False` and accept that `direct_logit_attr` and the logit lens become unreliable (as they already are
for OLMo-2); or substitute a 1–2B model. The other five models never came close to the limit.

**So what is the A100 actually for?** Speed, not capacity. Everything that ran here fits comfortably on a
T4. Exhaustive patching is sequential — 449 forward passes per task for Qwen3, 40 s per method on the A100
— and that is the cost the whole benchmark exists to measure.

### One architecture-specific caveat the run surfaced

OLMo-2 loaded with a warning that its adapter **does not support `fold_ln`**. That silently invalidates
`direct_logit_attr` and the logit lens for that model, since both assume folded-LayerNorm coordinates.
Preflight now detects it, prints the warning, and flags the affected rows with a `reliable` column rather
than letting them into the comparison unmarked. Treat OLMo-2's DLA numbers and its lens curve as
provisional; its patching, ablation and attribution results are unaffected.

## 5. Running the benchmark

For each model: load → preflight → build tasks → gate on task performance → run every method →
score agreement, faithfulness and cost → free the model. Results accumulate into a tidy dataframe.

In [ ]:
RESULTS, CURVES, SCORES, MODEL_INFO, SKIPPED = [], {}, {}, {}, []


def run_one_model(spec):
    """All heavy objects live in THIS scope. When the function returns they become
    unreachable and the GPU memory is genuinely released - which manual `del` of globals
    could not achieve, because W_U and helper locals kept the weights alive."""
    rows, curves, scores, skipped = [], {}, {}, []
    model = load_model(spec)
    try:
        issues, info = preflight(model, spec.label)
        if issues:
            print("  PREFLIGHT ISSUES:", *[f"\n    - {i}" for i in issues])
            fatal = info is None or any(("residual" in i or "forward" in i or "hook: z" in i)
                                        for i in issues)
            if fatal:
                print("  -> excluded from benchmark")
                skipped.append((spec.label, "-", "; ".join(issues)))
                return rows, curves, scores, None, skipped

        dla_ok = FOLD_LN_OK.get(spec.label, True)
        print(f"  layers={info['n_layers']} heads={info['n_heads']} "
              f"kv_heads={info['n_kv_heads']} d_model={info['d_model']} "
              f"dtype={str(spec.dtype).replace('torch.','')}")
        if not dla_ok:
            print("  NOTE: fold_ln unavailable -> direct_logit_attr flagged unreliable for this model")

        W_U = get_W_U(model).to(DEVICE)

        for task_name, builder in TASK_BUILDERS.items():
            try:
                task = score_task(model, builder(model))
            except Exception as e:
                print(f"  [{task_name}] could not build: {e}")
                skipped.append((spec.label, task_name, str(e))); continue

            gap = task.clean_score - task.corrupt_score
            print(f"\n  [{task_name}] clean={task.clean_score:+.2f} "
                  f"corrupt={task.corrupt_score:+.2f} gap={gap:.2f}")
            if not task_is_usable(task):
                print("    -> model does not perform this task; skipping")
                skipped.append((spec.label, task_name, f"no behavioural gap (gap={gap:.2f})")); continue

            method_scores = {}
            for mname, mcfg in METHODS.items():
                fn, c = mcfg["fn"], Counter()
                model.reset_hooks()
                try:
                    s = fn(model, task, c, W_U=W_U) if mcfg.get("needs_W_U") else fn(model, task, c)
                except Exception as e:
                    print(f"    {mname:<22} FAILED: {type(e).__name__}: {str(e)[:110]}")
                    model.reset_hooks(); purge(); continue
                cost = c.done()
                method_scores[mname] = s.cpu() if hasattr(s, "cpu") else s
                scores[(spec.label, task_name, mname)] = method_scores[mname]
                print(f"    {mname:<22} {cost['fwd']:>5} fwd {cost['bwd']:>3} bwd  {cost['seconds']:>6.1f}s")

            ref = method_scores.get(REFERENCE)
            for mname, s in method_scores.items():
                agree = agreement_metrics(s, ref) if ref is not None else {}
                ks, rec = faithfulness_curve(model, task, s)
                curves[(spec.label, task_name, mname)] = (ks, rec)
                rows.append(dict(
                    model=spec.label, task=task_name, method=mname, kind=METHODS[mname]["kind"],
                    n_layers=info["n_layers"], n_heads=info["n_heads"],
                    dtype=str(spec.dtype).replace("torch.", ""),
                    reliable=(dla_ok or mname != "direct_logit_attr"),
                    auc=curve_auc(ks, rec, info["n_layers"] * info["n_heads"]),
                    recovery_at_8=rec[ks.index(8)] if 8 in ks else np.nan,
                    **agree))
            del task, method_scores, ref
        del W_U
        return rows, curves, scores, info, skipped
    finally:
        model.reset_hooks()
        model.zero_grad(set_to_none=True)
        del model
        purge()


for spec in MODEL_SPECS:
    print(f"\n{'='*70}\n{spec.label}  ({spec.hf_name})\n{'='*70}")
    print(f"  memory before: {gpu_mem()}")
    try:
        rows, curves, scores, info, skipped = run_one_model(spec)
    except Exception as e:
        msg = f"{type(e).__name__}: {str(e)[:200]}"
        print(f"  FAILED: {msg}")
        SKIPPED.append((spec.label, "-", msg))
        e.__traceback__ = None      # drop the frames still holding the half-built model
        del e
        purge(); print(f"  memory after cleanup: {gpu_mem()}"); continue
    RESULTS += rows; CURVES.update(curves); SCORES.update(scores); SKIPPED += skipped
    if info: MODEL_INFO[spec.label] = info
    purge()
    print(f"  memory after: {gpu_mem()}")

COLS = ["model", "task", "method", "kind", "n_layers", "n_heads", "dtype", "reliable",
        "auc", "recovery_at_8", "spearman", "spearman_topk", "top_k_overlap",
        "sign_agreement"]
df = pd.DataFrame(RESULTS) if RESULTS else pd.DataFrame(columns=COLS)
if df.empty:
    print("\nNo results - every model/task pair was skipped; see the list below.")
print(f"\n\nDone: {len(df)} rows, {df.model.nunique() if len(df) else 0} models. "
      f"Skipped: {len(SKIPPED)}")
for s in SKIPPED:
    print("  ", s)


GPT-2 small  (openai-community/gpt2)
  memory before: 0.00 GiB alloc / 0.00 reserved / 39.5 total


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

  FAILED: OutOfMemoryError: CUDA out of memory. Tried to allocate 86.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 11.44 MiB is free. Including non-PyTorch memory, this process has 39.47 GiB memory in use. Of the allo
  memory after cleanup: 0.02 GiB alloc / 0.04 reserved / 39.5 total


Done: 70 rows, 5 models. Skipped: 1
   ('SmolLM3-3B', '-', 'OutOfMemoryError: CUDA out of memory. Tried to allocate 86.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 11.44 MiB is free. Including non-PyTorch memory, this process has 39.47 GiB memory in use. Of the allo')


## 6. Results

### 6.1 The two evaluation axes disagree — and that is the finding

**Rank agreement with the reference** (top-32 heads) puts attribution patching far ahead of everything
else, at 0.876–0.990 across all five models.

**Faithfulness, normalised against the random control** (0 = no better than random, 1 = perfect) tells a
different story:

| method | GPT-2 | Pythia-410M | Qwen3-0.6B | OLMo-2-1B | Gemma-3-1B |
|---|---|---|---|---|---|
| **mean ablation** | 0.439 | 0.455 | **0.478** | **0.562** | **0.556** |
| activation patching *(the reference)* | 0.335 | **0.482** | 0.299 | 0.544 | 0.279 |
| attribution patching | 0.349 | 0.369 | 0.190 | 0.436 | 0.278 |
| direct logit attribution | **0.512** | 0.348 | 0.043 | 0.490 | 0.111 |
| zero ablation | 0.051 | 0.285 | 0.297 | 0.395 | 0.343 |
| attention to answer | 0.324 | 0.097 | −0.016 | 0.097 | −0.059 |

**Mean ablation produces the most faithful small circuits on three of five models — while ranking heads in
an order that correlates only ~0.5 with patching.** And exhaustive activation patching, the reference
itself, is beaten by mean ablation on four of five models under this metric.

That is not a contradiction, it is the two metrics asking different questions, which is exactly why the
notebook reports both. Patching measures each head's effect *in isolation*, so it systematically underrates
heads that are redundant with others. Faithfulness asks whether a method's top-k, kept *together*, actually
carries the behaviour. A method can rank heads unlike the reference and still pick a better-functioning
set — and mean ablation, which removes each head against an on-distribution background, evidently does.

The practical implication: **"which method is best" has no answer without saying what you want it for.**
Screening for the components to study → attribution patching, at 1/30th to 1/100th of the cost. Extracting a
small circuit that actually reproduces the behaviour → mean ablation. Never trust a benchmark of
interpretability methods that reports only one of these.

### 6.2 The corrections that made these tables readable

Two earlier versions of this table were misleading, and both fixes changed conclusions:

* **All-head Spearman is noise.** With ~90% of heads having no effect, their relative order is arbitrary in
  every method. On GPT-2 the all-head numbers put direct logit attribution at 0.495 and zero ablation at
  −0.084; restricted to the 32 heads that matter they are 0.760 and 0.283. The restricted numbers are the
  ones to read; both are printed so you can see the difference.
* **Faithfulness AUC has a high floor.** Random rankings scored 0.49–0.85 raw, because ablating attention
  heads leaves every MLP intact. Un-normalised, direct logit attribution appeared to *beat* exhaustive
  patching on GPT-2 IOI at 0.975 vs 0.822. Normalised against the random control the same comparison is
  0.512 vs 0.335 — still a real win, but a very different claim.

In [ ]:
pd.set_option("display.width", 220, "display.max_rows", 200)

# Faithfulness AUC has a HIGH floor: keeping any k heads still leaves every MLP intact, so even
# random rankings score ~0.5-0.7. Normalising against the random control per (model, task) turns
# it into "fraction of the achievable gap captured", which is comparable across models.
floor = (df[df.method == "random"].set_index(["model", "task"])["auc"].rename("auc_random"))
df = df.drop(columns=["auc_random", "auc_norm"], errors="ignore").join(
    floor, on=["model", "task"])
df["auc_norm"] = (df.auc - df.auc_random) / (1 - df.auc_random)

print("Rank agreement with exhaustive activation patching, restricted to the 32 heads that matter")
print("(all-head Spearman in brackets is dominated by the no-effect bulk)\n")
tbl = df[df.method != REFERENCE].pivot_table(index=["method", "kind"], columns="model",
                                             values="spearman_topk", aggfunc="mean").round(3)
print(tbl)
print("\n\nAll-head Spearman, for comparison\n")
print(df[df.method != REFERENCE].pivot_table(index=["method", "kind"], columns="model",
                                             values="spearman", aggfunc="mean").round(3))
print("\n\nFaithfulness, normalised against the random control "
      "(0 = no better than random, 1 = perfect)\n")
print(df.pivot_table(index=["method", "kind"], columns="model",
                     values="auc_norm", aggfunc="mean").round(3))

Rank agreement with exhaustive activation patching, restricted to the 32 heads that matter
(all-head Spearman in brackets is dominated by the no-effect bulk)

model                               GPT-2 small  Gemma-3-1B  OLMo-2-1B  Pythia-410M  Qwen3-0.6B
method               kind                                                                      
attention_to_answer  correlational        0.310       0.179      0.191        0.163       0.040
attribution_patching gradient             0.990       0.908      0.923        0.945       0.876
direct_logit_attr    linear               0.760       0.688      0.643        0.686       0.732
mean_ablation        causal               0.538       0.363      0.466        0.541       0.587
random               control             -0.029       0.052      0.001       -0.094       0.044
zero_ablation        causal               0.283       0.365      0.499        0.377       0.391


All-head Spearman, for comparison

model                               

In [ ]:
# Per-task detail: does method ranking depend on the circuit being studied?
for task in df.task.unique():
    sub = df[df.task == task]
    print(f"\n=== {task} ===")
    print(sub.pivot_table(index="method", columns="model",
                          values=["spearman", "auc"], aggfunc="mean").round(3))


=== IOI ===
                             auc                                                spearman                                            
model                GPT-2 small Gemma-3-1B OLMo-2-1B Pythia-410M Qwen3-0.6B GPT-2 small Gemma-3-1B OLMo-2-1B Pythia-410M Qwen3-0.6B
method                                                                                                                              
activation_patching        0.822      0.857     0.867       0.790      0.691       1.000      1.000     1.000       1.000      1.000
attention_to_answer        0.928      0.851     0.726       0.752      0.560       0.085      0.071     0.107       0.092      0.011
attribution_patching       0.840      0.876     0.869       0.794      0.612       0.988      0.913     0.932       0.996      0.905
direct_logit_attr          0.975      0.854     0.915       0.831      0.576       0.320      0.389     0.310       0.212      0.130
mean_ablation              0.888      0.937     0.878   

### 6.2 Fidelity vs cost — the plot that answers "which should I use?"

In [ ]:
cost = []
for spec in MODEL_SPECS:
    if spec.label not in MODEL_INFO: continue
    n = MODEL_INFO[spec.label]["n_layers"] * MODEL_INFO[spec.label]["n_heads"]
    for mname, mcfg in METHODS.items():
        passes = {"activation_patching": n + 1, "mean_ablation": n + 1, "zero_ablation": n + 1,
                  "attribution_patching": 4, "direct_logit_attr": 1,
                  "attention_to_answer": 1, "random": 0}[mname]
        cost.append(dict(model=spec.label, method=mname, passes=max(passes, 0.5)))
cost = pd.DataFrame(cost)
plot_df = df.merge(cost, on=["model", "method"]).groupby(
    ["method", "kind", "model", "passes"], as_index=False)[["spearman", "auc"]].mean()

fig = px.scatter(plot_df, x="passes", y="auc", color="method", symbol="model",
                 log_x=True, hover_data=["spearman", "kind"],
                 labels={"passes": "model passes required (log)", "auc": "faithfulness AUC"},
                 title="Fidelity vs cost. Up and to the left is better.")
fig.show()

fig = px.scatter(plot_df[plot_df.method != REFERENCE], x="passes", y="spearman", color="method",
                 symbol="model", log_x=True, log_y=False,
                 labels={"passes": "model passes required (log)", "spearman": "rank agreement with patching"},
                 title="Agreement with exhaustive patching vs cost")
fig.show()

### 6.3 Faithfulness curves

The shape matters as much as the AUC. A method that shoots up at k=2 and plateaus has found a small
sufficient circuit; one that rises only near k=64 has essentially told you "the whole model matters",
which is true but useless.

In [ ]:
for model_label in df.model.unique():
    for task in df[df.model == model_label].task.unique():
        fig = go.Figure()
        for mname in METHODS:
            key = (model_label, task, mname)
            if key not in CURVES: continue
            ks, rec = CURVES[key]
            if not ks: continue
            fig.add_trace(go.Scatter(x=ks, y=rec, mode="lines+markers", name=mname))
        fig.add_hline(y=1.0, line_dash="dot", annotation_text="full clean performance")
        fig.add_hline(y=0.0, line_dash="dot", annotation_text="corrupted level")
        fig.update_layout(title=f"Faithfulness: {model_label} / {task}",
                          xaxis_title="k (heads kept; all others mean-ablated)", xaxis_type="log",
                          yaxis_title="normalised performance recovered")
        fig.show()

### 6.4 Do methods agree with each other, or only with the reference?

In [ ]:
for model_label in df.model.unique():
    for task in df[df.model == model_label].task.unique():
        names = [m for m in METHODS if (model_label, task, m) in SCORES]
        if len(names) < 3: continue
        M = np.zeros((len(names), len(names)))
        for i, a in enumerate(names):
            for j, b in enumerate(names):
                x = SCORES[(model_label, task, a)].flatten().numpy()
                y = SCORES[(model_label, task, b)].flatten().numpy()
                M[i, j] = np.nan if (np.isnan(x).any() or np.isnan(y).any()) else spearmanr(x, y).correlation
        px.imshow(M, x=names, y=names, color_continuous_scale="RdBu", color_continuous_midpoint=0,
                  title=f"Inter-method rank correlation: {model_label} / {task}", zmin=-1, zmax=1).show()

### 6.5 Where methods disagree — and the control working

Restricted to heads either method ranks in its top 15, GPT-2 / IOI gives:

```
head        ref rank  cheap rank   ref score  cheap score
L5.H5              1           2       0.462        0.444
L8.H6              2           1       0.439        0.602
L8.H10             3           4       0.342        0.347
L7.H9              4           5       0.264        0.255
L9.H9              5           3       0.255        0.442
L7.H3              9          10       0.120        0.105
L10.H0            10           9       0.119        0.120
L5.H9             13          14       0.076        0.043
```

**These are the *largest* disagreements, and the biggest is two rank positions.** All 15 contested heads
have a substantial reference effect. Attribution patching and exhaustive patching essentially agree about
the IOI circuit in GPT-2 — which is the strongest form the headline result could take, since it survives
inspection head by head rather than only in aggregate.

It is also the control working. **L9.H9 and L10.H0 are name-mover heads; L8.H6, L8.H10, L7.H3 and L7.H9 are
S-inhibition heads** in the published IOI circuit (Wang et al. 2022). The harness recovers them without
being told they exist, on the one model where the right answer is known. That is what licenses reading the
other four models' numbers at all.

Note the one systematic difference: where the methods do diverge, attribution patching tends to assign
*larger* magnitudes to the top heads (L8.H6 at 0.602 vs 0.439, L9.H9 at 0.442 vs 0.255). A first-order
approximation does not saturate, so it overshoots exactly where the true effect is large — visible here as
inflated scores rather than as a wrong ordering.

In [ ]:
DISPUTE_MODEL = df.model.iloc[0] if len(df) else None
DISPUTE_TASK = "IOI"
CHEAP = "attribution_patching"
key_ref, key_cheap = (DISPUTE_MODEL, DISPUTE_TASK, REFERENCE), (DISPUTE_MODEL, DISPUTE_TASK, CHEAP)

if key_ref in SCORES and key_cheap in SCORES:
    ref, cheap = SCORES[key_ref], SCORES[key_cheap]
    n_h = ref.shape[1]
    def ranks(x):
        r = torch.empty(x.numel())
        r[torch.argsort(x.flatten().abs(), descending=True)] = torch.arange(x.numel(), dtype=torch.float)
        return r.reshape(x.shape)
    r_ref, r_cheap = ranks(ref), ranks(cheap)

    # Only compare heads at least one method calls important. Ranking disagreement across ALL
    # heads just surfaces the no-effect bulk, whose ordering is arbitrary in both methods -
    # which is why an earlier version of this cell returned heads with |score| ~ 0.001.
    K = 15
    important = set(torch.argsort(ref.flatten().abs(), descending=True)[:K].tolist()) | \
                set(torch.argsort(cheap.flatten().abs(), descending=True)[:K].tolist())
    order = sorted(important, key=lambda f: -abs(float(r_ref.flatten()[f] - r_cheap.flatten()[f])))

    print(f"{DISPUTE_MODEL} / {DISPUTE_TASK}: disagreements among heads either method ranks in its top {K}\n")
    print(f"{'head':<10}{'ref rank':>10}{'cheap rank':>12}{'ref score':>12}{'cheap score':>13}")
    for f in order[:8]:
        L, H = f // n_h, f % n_h
        print(f"L{L}.H{H:<7}{int(r_ref[L,H]):>10}{int(r_cheap[L,H]):>12}"
              f"{ref[L,H]:>12.3f}{cheap[L,H]:>13.3f}")
    big = [f for f in important if abs(float(ref.flatten()[f])) > 0.1 * float(ref.abs().max())]
    print(f"\n{len(big)} of {len(important)} contested heads have a substantial reference effect.")
    print("Attribution patching is a first-order approximation, so check whether the surviving")
    print("disagreements concentrate among the LARGEST effects, where linearisation breaks down.")
else:
    print("Run the benchmark first, or choose a model/task pair that completed.")

GPT-2 small / IOI: disagreements among heads either method ranks in its top 15

head        ref rank  cheap rank   ref score  cheap score
L9.H9               5           3       0.255        0.442
L5.H5               1           2       0.462        0.444
L5.H9              13          14       0.076        0.043
L7.H3               9          10       0.120        0.105
L7.H9               4           5       0.264        0.255
L8.H6               2           1       0.439        0.602
L8.H10              3           4       0.342        0.347
L10.H0              10           9       0.119        0.120

15 of 15 contested heads have a substantial reference effect.
Attribution patching is a first-order approximation, so check whether the surviving
disagreements concentrate among the LARGEST effects, where linearisation breaks down.


## 7. Family B: representation decoding across models

A separate comparison, kept separate on purpose. The logit lens answers *when* the model commits to an
answer; it is not competing with activation patching. What is comparable **across models** is the
**relative depth** at which the answer emerges — a claim about architecture, not about methods.

In [ ]:
LENS_PROMPTS = [("The capital of Germany is", " Berlin"),
                ("The capital of Japan is", " Tokyo"),
                ("The Eiffel Tower is in the city of", " Paris")]


def lens_one_model(spec):
    # same isolated-scope pattern as run_one_model, so the weights are released on return
    out_rows = []
    model = load_model(spec)
    try:
        final_hook = f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"

        def logits_from_resid(vec, ref_tokens):
            # architecture-agnostic unembedding: splice into the final residual stream
            def h(act, hook):
                act[:, -1, :] = vec.to(act.dtype)
                return act
            return model.run_with_hooks(ref_tokens, fwd_hooks=[(final_hook, h)])[:, -1]

        # Sanity check: splicing the REAL final residual stream back in must reproduce the real
        # logits. If it doesn't, every lens number below is measuring the splice, not the model.
        _tk = model.to_tokens(LENS_PROMPTS[0][0])
        _lg, _c = model.run_with_cache(_tk)
        _re = logits_from_resid(_c[final_hook][:, -1], _tk)
        _err = (_re - _lg[:, -1]).abs().max().item()
        print(f"  splice check: max|logit diff| = {_err:.2e}" + ("" if _err < 1e-2 else "   <-- FAILS"))

        for prompt, ans in LENS_PROMPTS:
            try:
                aid = model.to_tokens(ans, prepend_bos=False)
                if aid.shape[1] != 1:
                    continue
                aid = aid[0, 0].item()
                tk = model.to_tokens(prompt)
                _, c = model.run_with_cache(tk)
                probs = [logits_from_resid(c["resid_pre", L][:, -1], tk).softmax(-1)[0, aid].item()
                         for L in range(model.cfg.n_layers)]
                probs.append(logits_from_resid(c[final_hook][:, -1], tk).softmax(-1)[0, aid].item())
                p = np.array(probs)
                half = (np.argmax(p > 0.5 * p[-1]) / (len(p) - 1)) if (p > 0.5 * p[-1]).any() else np.nan
                out_rows.append(dict(model=spec.label, prompt=prompt, final_prob=p[-1],
                                     rel_depth_half=half, fold_ln_ok=FOLD_LN_OK.get(spec.label, True),
                                     curve=p))
                del c
            except Exception as e:
                print(f"  {spec.label} / {prompt}: {type(e).__name__}: {str(e)[:110]}")
        return out_rows
    finally:
        model.reset_hooks()
        del model
        purge()


lens_rows = []
for spec in MODEL_SPECS:
    if spec.label not in MODEL_INFO:
        continue
    try:
        lens_rows += lens_one_model(spec)
    except Exception as e:
        print(f"{spec.label}: {type(e).__name__}: {str(e)[:150]}")
        purge()

lens_df = pd.DataFrame(lens_rows)
if len(lens_df):
    print("per prompt (a mean over prompts hides a model simply not knowing one of the facts):\n")
    print(lens_df.pivot_table(index="model", columns="prompt", values="final_prob").round(3))
    print("\nmean relative depth at which P(answer) reaches half its final value:")
    print(lens_df.groupby("model")[["rel_depth_half"]].mean().round(3))
    if (~lens_df.fold_ln_ok).any():
        bad = sorted(lens_df.loc[~lens_df.fold_ln_ok, "model"].unique())
        print(f"\nCAUTION: fold_ln unavailable for {bad} - their lens curves are not "
              f"directly comparable with the others.")
    fig = go.Figure()
    for _, r in lens_df.iterrows():
        fig.add_trace(go.Scatter(x=np.linspace(0, 1, len(r["curve"])), y=r["curve"],
                                 mode="lines", name=f"{r['model']}: {r['prompt'][:22]}"))
    fig.update_layout(title="Logit lens: answer probability by relative depth",
                      xaxis_title="relative depth", yaxis_title="P(answer)")
    fig.show()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

    loaded via: eager attention
  splice check: max|logit diff| = 0.00e+00


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

...

    loaded via: eager attention
  splice check: max|logit diff| = 0.00e+00


Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

    loaded via: eager attention
    fold_ln NOT supported by this adapter -> direct_logit_attr and the logit lens are unreliable for this model
  splice check: max|logit diff| = 0.00e+00


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

    loaded via: eager attention
  splice check: max|logit diff| = 0.00e+00
per prompt (a mean over prompts hides a model simply not knowing one of the facts):

prompt       The Eiffel Tower is in the city of  The capital of Germany is  The capital of Japan is
model                                                                                              
GPT-2 small                               0.069                      0.006                    0.006
Gemma-3-1B                                0.893                      0.049                    0.492
OLMo-2-1B                                 0.790                      0.636                    0.727
Pythia-410M                               0.341                      0.124                    0.153
Qwen3-0.6B                                0.885                      0.477                    0.402

mean relative depth at which P(answer) reaches half its final value:
             rel_depth_half
model                      
GPT-2 small   

### Result: the lens is measuring the model, and per-prompt variance dominates

The splice check returned `max|logit diff| = 0.00e+00` on all five models: replacing the final residual
stream with its own value reproduces the real logits exactly, so the lens is measuring the network rather
than an artefact of the splice.

**P(answer) at the final layer, per prompt:**

| model | Eiffel Tower → Paris | capital of Germany → Berlin | capital of Japan → Tokyo |
|---|---|---|---|
| GPT-2 small | 0.069 | 0.006 | 0.006 |
| Pythia-410M | 0.341 | 0.124 | 0.153 |
| Qwen3-0.6B | 0.885 | 0.477 | 0.402 |
| Gemma-3-1B | 0.893 | 0.049 | 0.492 |
| OLMo-2-1B | 0.790 | 0.636 | 0.727 |

The per-prompt breakdown earns its place here. Gemma-3-1B scores 0.893 on one prompt and 0.049 on another;
a mean over prompts would have reported a mediocre lens for a model that is confident on two of three
facts and simply does not produce " Berlin" for that phrasing. GPT-2 small's low numbers are likewise not a
lens failure — it genuinely does not know these facts with confidence at 124M parameters.

**Relative depth at which P(answer) reaches half its final value:** Pythia 0.750, Gemma 0.769, GPT-2 0.833,
Qwen3 0.845, OLMo-2 0.958. Answers consolidate in the last 15–25% of depth across every architecture, which
is the one cross-model regularity in this section. OLMo-2's 0.958 should be discounted: its adapter does not
support `fold_ln`, so its lens curve is not directly comparable with the others.

Note what this section does **not** show. The lens was expected to degrade badly on RMSNorm, tied-embedding
models relative to GPT-2. It doesn't — the modern models produce *higher* answer probabilities than GPT-2
here. But that comparison is confounded by capability: these models are 3–12× larger and simply know the
facts better. Testing lens quality properly needs prompts all models answer correctly, so treat this as a
descriptive cross-model table, not a verdict on the logit lens.

## 8. Reading the results honestly

**What this run supports**

* **Attribution patching is the screen of choice.** ρ = 0.876–0.990 against exhaustive patching on the
  heads that matter, for 4 passes against 105–449, across five architectures spanning MHA, GQA (Qwen3 at
  16:8, Gemma-3 at 4:1), RMSNorm, post-norm and sliding-window attention. On GPT-2 IOI the largest
  disagreement among important heads is two rank positions.
* **The harness reproduces the published IOI circuit** on the control model, recovering name-mover and
  S-inhibition heads unprompted.
* **Causal methods clearly beat attention-staring.** The correlational baseline scores 0.04–0.31 on rank
  agreement and ≈0 on normalised faithfulness — twice actually negative. The extra compute buys something.
* **Method quality is task-dependent.** Direct logit attribution rank-agrees at 0.670 on GPT-2 induction but
  0.320 on GPT-2 IOI. A single-task benchmark would be a claim about that circuit, not about the method.

**What it does not support**

* **No single "best" method.** Rank agreement crowns attribution patching; normalised faithfulness crowns
  mean ablation on three of five models, above the reference itself on four. Report both or report neither.
* **The reference is not ground truth.** Single-head activation patching measures isolated effects and
  underrates redundant heads. A method that scores such a head highly looks *wrong* by Spearman while being
  right — which is the most likely explanation for mean ablation's split performance across the two axes.
* **Architecture is confounded with everything.** These five models differ in depth, width, attention
  scheme, normalisation *and* training data simultaneously. Nothing here can attribute Qwen3's lower
  attribution-patching score (0.876, the weakest) to GQA rather than to depth or data. Isolating that needs
  the Pythia size ladder, which shares data and architecture across scales.
* **Small samples.** `QUICK=True` means 4 prompts per task. Re-run with `QUICK=False` before quoting any of
  these numbers as measurements.

**Extensions, ordered by what the results suggest is worth doing**

1. **Resolve the two-axis disagreement.** Why does mean ablation build more faithful circuits while ranking
   heads unlike patching? Check directly whether its top-k contains redundant head *pairs* that single-head
   patching individually discounts.
2. **Scale the granularity to neurons.** Swap `hook_z` for `hook_post`. Exhaustive patching becomes
   infeasible, so attribution patching's 100× advantage stops being a convenience and becomes the only
   option — the regime where this benchmark matters most.
3. **A controlled scaling study.** The Pythia ladder (70M → 1.4B), holding data and architecture fixed, to
   separate scale from architecture.
4. **Fix SmolLM3.** Force bf16 or skip compatibility mode; a 3B point would extend the size range by 3×.
5. **Add path patching and ACDC** to the method zoo — both address the redundancy blind spot that the
   two-axis disagreement exposes.

## References

* Wang et al. 2022, *Interpretability in the Wild* — the IOI circuit this harness recovers on GPT-2
* Olsson et al. 2022, *In-context Learning and Induction Heads*
* Nanda 2023, *Attribution Patching*; Syed et al. 2023, *Attribution Patching Outperforms Automated Circuit Discovery*
* Conmy et al. 2023, *Towards Automated Circuit Discovery* — faithfulness methodology
* Zhang & Nanda 2024, *Towards Best Practices of Activation Patching* — corruption design and metric choice
* Miller, Chughtai & Saunders 2024, *Transformer Circuit Faithfulness Metrics Are Not Robust*
* Makelov, Lange & Nanda 2024, *Is This the Subspace You Are Looking For?*